# Caracal s07.A - Setup tokenizer + base model

Pedro session 1/7. ~12h Kaggle T4 x2 (na verdade ~1h - stage rápido).

Deliverable: pedroafonso2/caracal-s07-base-tokens (HF dataset)

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'accelerate>=1.0.0' kaggle

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic', 'https://github.com/iterate-labs-ai/caracal-1.git', '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
rev = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'cloned {rev}')

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()} n_gpu: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)}')

In [ ]:
import subprocess
subprocess.run([
    'python', 'train/s07/00_setup_tokenizer.py',
    '--base-model', 'Qwen/Qwen2.5-3B-Instruct',
    '--s05-adapter', 'pedroafonso2/caracal-base-3b-s05',
    '--out-dir', '/kaggle/working/caracal-s07-base',
], check=True)

In [ ]:
import json, subprocess
from pathlib import Path
pub = Path('/kaggle/working/caracal-s07-base-published')
pub.mkdir(exist_ok=True)
for f in Path('/kaggle/working/caracal-s07-base').iterdir():
    if f.is_file():
        subprocess.run(['cp', str(f), str(pub / f.name)], check=True)
meta = {'title': 'Caracal s07 base tokens', 'id': 'pedroafonso2/caracal-s07-base-tokens', 'licenses': [{'name': 'Apache-2.0'}]}
(pub / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(pub), '--public'], capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(['kaggle', 'datasets', 'version', '-p', str(pub), '-m', 's07.A output'], check=True)